# EMMa Counterfactual Candidate Pool

In [ ]:
import json
import sys
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path().resolve().parent.parent
sys.path.insert(0, str(DATA_DIR))

from emma import EmmaDataset
from models.density_lut import load_density_lut

In [ ]:
DATA_CONFIG = DATA_DIR / "configs" / "data.json"
with DATA_CONFIG.open(encoding="utf-8") as f:
    data_config = json.load(f)

DATASET_ROOT = Path(data_config["emma_root"])
TRAIN_ASINS_PATH = Path(data_config["asins"]["train"])
SINGLE_MATERIAL_ASINS_PATH = Path(data_config["single_material_asins"])
SINGLE_MATERIAL_MATERIALS_PATH = Path(data_config["single_material_materials"])
DENSITY_LUT_PATH = Path(data_config["density_lut"])
DENSITY_RANGE_LUT_PATH = Path(data_config["density_range_lut"])
COUNTERFACTUAL_CANDIDATE_POOL_PATH = (DATA_DIR / "artifacts" / "counterfactual_candiate_pool" / "results.csv")

In [ ]:
ds = EmmaDataset(DATASET_ROOT)

material_names = list(ds.material_names)

material_densities = np.asarray(
    load_density_lut(
        DENSITY_LUT_PATH,
        material_names,
        unit=data_config.get("density_lut_unit", "g_cm3"),
    ),
    dtype=float,
)

train_asins = set(np.load(TRAIN_ASINS_PATH, allow_pickle=False).astype(str).reshape(-1))
single_asins = (np.load(SINGLE_MATERIAL_ASINS_PATH, allow_pickle=False).astype(str).reshape(-1))
single_materials = (np.load(SINGLE_MATERIAL_MATERIALS_PATH, allow_pickle=False).astype(str).reshape(-1))

In [ ]:
def material_key(name):
    return " ".join(str(name).strip().lower().replace("_", " ").split())

with DENSITY_RANGE_LUT_PATH.open(encoding="utf-8") as f:
    density_ranges = {
        material_key(name): density_range
        for name, density_range in json.load(f).items()
    }

key_to_idx = {material_key(name): i for i, name in enumerate(material_names)}

In [ ]:
parent = np.full(len(material_names), -1, dtype=int)
for child_name, parent_name in ds.material_hierarchy["parent"].items():
    child_idx = key_to_idx.get(material_key(child_name))
    parent_idx = (
        None
        if parent_name is None
        else key_to_idx.get(material_key(parent_name))
    )

    if child_idx is not None and parent_idx is not None:
        parent[child_idx] = parent_idx

ancestors = []
depths = np.zeros(len(material_names), dtype=int)
roots = np.zeros(len(material_names), dtype=int)

for i in range(len(material_names)):
    chain = []
    current = i
    seen = set()

    while parent[current] >= 0 and current not in seen:
        seen.add(current)
        current = int(parent[current])
        chain.append(current)

    ancestors.append(tuple(chain))
    depths[i] = len(chain)
    roots[i] = current

In [ ]:
donors_by_material = {}

for asin, material in zip(single_asins, single_materials):
    if asin not in train_asins:
        continue

    j = key_to_idx.get(material_key(material))
    if j is None:
        continue

    record = ds.get_record(asin)
    prior = np.asarray(
        ds.get_material_probs(record),
        dtype=np.float32,
    ).reshape(-1)[:len(material_names)]

    donors_by_material.setdefault(j, []).append(prior)

print(
    f"{'Training single-material donors':<32} "
    f"ASINs: {sum(map(len, donors_by_material.values())):,}"
)
print(f"{'Materials with donors':<32} Materials: {len(donors_by_material):,}")

In [ ]:
candidate_rows = []

for j, donor_vectors in donors_by_material.items():
    vectors = np.stack(donor_vectors)
    proto = np.median(vectors, axis=0).astype(np.float32)

    density_info = density_ranges.get(material_key(material_names[j]), {})
    low = density_info.get("min", np.nan)
    high = density_info.get("max", np.nan)
    mid = (
        (low + high) / 2
        if np.isfinite(low) and np.isfinite(high)
        else np.nan
    )
    span = (
        (high - low) / mid
        if np.isfinite(mid) and mid > 0
        else np.nan
    )

    allowed = {j, *ancestors[j]}
    offbranch = [
        k
        for k in range(len(material_names))
        if k not in allowed
    ]

    if offbranch:
        donor_offbranch_max = vectors[:, offbranch].max(axis=1)
        prototype_offbranch_max = proto[offbranch].max()
        prototype_offbranch_sum = proto[offbranch].sum()
    else:
        donor_offbranch_max = np.zeros(len(vectors), dtype=np.float32)
        prototype_offbranch_max = 0.0
        prototype_offbranch_sum = 0.0

    parent_idx = int(parent[j])
    path = [*reversed(ancestors[j]), j]

    candidate_rows.append(
        {
            "material_index": j,
            "material": material_names[j],
            "root_material": material_names[int(roots[j])],
            "parent_material": (
                material_names[parent_idx]
                if parent_idx >= 0
                else ""
            ),
            "hierarchy_path": " > ".join(material_names[k] for k in path),
            "material_depth": int(depths[j]),
            "n_train_donors": len(vectors),
            "density_min": low,
            "lut_density": material_densities[j],
            "density_max": high,
            "density_relative_span": span,
            "prototype_target_prior": proto[j],
            "prototype_offbranch_max": prototype_offbranch_max,
            "prototype_offbranch_sum": prototype_offbranch_sum,
            "donor_offbranch_max_p50": np.median(donor_offbranch_max),
            "donor_offbranch_max_p90": np.quantile(
                donor_offbranch_max,
                0.90,
            ),
        }
    )

counterfactual_candidate_pool = (
    pd.DataFrame(candidate_rows)
    .sort_values("lut_density")
    .reset_index(drop=True)
)

In [ ]:
COUNTERFACTUAL_CANDIDATE_POOL_PATH.parent.mkdir(parents=True, exist_ok=True)
counterfactual_candidate_pool.to_csv(
    COUNTERFACTUAL_CANDIDATE_POOL_PATH,
    index=False,
)

print(
    f"Saved {len(counterfactual_candidate_pool)} rows to: "
    f"{COUNTERFACTUAL_CANDIDATE_POOL_PATH}"
)

counterfactual_candidate_pool.round(4)